# DSA 07: Sorting, Searching and Greedy
## Ordering, Finding and Making Optimal Choices

**What you'll learn:**
- All major sorting algorithms (bubble, merge, quick, heap sort)
- Python's built-in sort (Timsort) and custom comparators
- Binary search (3 templates: exact, lower bound, upper bound)
- Binary search on answer (minimize/maximize problems)
- Greedy algorithms and when they work
- 15+ practice problems with solutions

**Prerequisites:** DSA_06 (Recursion, Backtracking, DP)
**Estimated Time:** 5-6 hours

---

---
# Section 1: Sorting Algorithms

## The Big Picture

| Algorithm | Time (avg) | Time (worst) | Space | Stable? |
|-----------|-----------|-------------|-------|---------|
| Bubble Sort | O(n^2) | O(n^2) | O(1) | Yes |
| Selection Sort | O(n^2) | O(n^2) | O(1) | No |
| Insertion Sort | O(n^2) | O(n^2) | O(1) | Yes |
| Merge Sort | O(n log n) | O(n log n) | O(n) | Yes |
| Quick Sort | O(n log n) | O(n^2) | O(log n) | No |
| Heap Sort | O(n log n) | O(n log n) | O(1) | No |
| **Timsort** (Python) | O(n log n) | O(n log n) | O(n) | Yes |

**Stable sort**: Equal elements maintain their original relative order.

## When to Use Each

- **Timsort** (Python's `sorted()`): Always use this in practice
- **Merge Sort**: When stability matters, guaranteed O(n log n)
- **Quick Sort**: Average fastest in practice, but O(n^2) worst case
- **Insertion Sort**: Small arrays (< 20 elements), nearly sorted data
- **Heap Sort**: When O(1) extra space is required

In [0]:
# Sorting algorithms from scratch
import time, random

def bubble_sort(arr):
    """O(n^2) -- compare adjacent pairs, bubble largest to end."""
    arr = arr[:]
    n = len(arr)
    for i in range(n):
        for j in range(n - 1 - i):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr

def merge_sort(arr):
    """O(n log n) -- divide and conquer."""
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

def quick_sort(arr):
    """O(n log n) average -- partition around pivot."""
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return quick_sort(left) + middle + quick_sort(right)

print("SORTING ALGORITHMS")
print("=" * 60)

# Correctness check
arr = [64, 34, 25, 12, 22, 11, 90]
print(f"Input: {arr}")
print(f"Bubble sort: {bubble_sort(arr)}")
print(f"Merge sort:  {merge_sort(arr)}")
print(f"Quick sort:  {quick_sort(arr)}")
print(f"Python sort: {sorted(arr)}")

# Performance comparison
random.seed(42)
large_arr = [random.randint(1, 10000) for _ in range(5000)]

print(f"\nPerformance on {len(large_arr)} elements:")
for name, func in [("Bubble", bubble_sort), ("Merge", merge_sort),
                    ("Quick", quick_sort), ("Python", sorted)]:
    start = time.perf_counter()
    func(large_arr)
    t = time.perf_counter() - start
    print(f"  {name:8}: {t:.4f}s")


---
# Section 2: Binary Search

## The Three Templates

Binary search is simple in concept but tricky in implementation.
There are 3 common templates:

```python
# Template 1: Find exact target
def binary_search(nums, target):
    left, right = 0, len(nums) - 1
    while left <= right:
        mid = (left + right) // 2
        if nums[mid] == target:
            return mid
        elif nums[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1

# Template 2: Find leftmost position (lower bound)
def lower_bound(nums, target):
    left, right = 0, len(nums)
    while left < right:
        mid = (left + right) // 2
        if nums[mid] < target:
            left = mid + 1
        else:
            right = mid
    return left

# Template 3: Binary search on answer
# "Find minimum X such that condition(X) is True"
def binary_search_answer(lo, hi, condition):
    while lo < hi:
        mid = (lo + hi) // 2
        if condition(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo
```

In [0]:
# Binary search patterns
print("BINARY SEARCH PATTERNS")
print("=" * 60)

# Template 1: Exact search
def binary_search(nums, target):
    left, right = 0, len(nums) - 1
    while left <= right:
        mid = (left + right) // 2
        if nums[mid] == target:
            return mid
        elif nums[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1

nums = [1, 3, 5, 7, 9, 11, 13, 15]
print(f"\n1. Exact search in {nums}:")
for target in [7, 6, 1, 15]:
    print(f"   Search {target}: index={binary_search(nums, target)}")

# Template 2: Search in rotated sorted array
def search_rotated(nums, target):
    left, right = 0, len(nums) - 1
    while left <= right:
        mid = (left + right) // 2
        if nums[mid] == target:
            return mid
        if nums[left] <= nums[mid]:  # Left half is sorted
            if nums[left] <= target < nums[mid]:
                right = mid - 1
            else:
                left = mid + 1
        else:  # Right half is sorted
            if nums[mid] < target <= nums[right]:
                left = mid + 1
            else:
                right = mid - 1
    return -1

rotated = [4, 5, 6, 7, 0, 1, 2]
print(f"\n2. Search in rotated array {rotated}:")
for target in [0, 3, 7]:
    print(f"   Search {target}: index={search_rotated(rotated, target)}")

# Template 3: Binary search on answer
def min_eating_speed(piles, h):
    """Koko eating bananas -- find minimum speed."""
    def can_finish(speed):
        return sum((p + speed - 1) // speed for p in piles) <= h

    left, right = 1, max(piles)
    while left < right:
        mid = (left + right) // 2
        if can_finish(mid):
            right = mid
        else:
            left = mid + 1
    return left

piles = [3, 6, 7, 11]
h = 8
print(f"\n3. Minimum eating speed (piles={piles}, h={h}):")
print(f"   Minimum speed: {min_eating_speed(piles, h)}")


---
# Section 3: Greedy Algorithms

## What is Greedy?

A greedy algorithm makes the **locally optimal choice** at each step,
hoping it leads to the globally optimal solution.

**When greedy works:**
- The problem has **greedy choice property**: local optimal = global optimal
- The problem has **optimal substructure**: optimal solution contains optimal subproblems

**Classic greedy problems:**
- Activity selection (maximize non-overlapping intervals)
- Huffman coding (optimal prefix codes)
- Dijkstra's shortest path
- Fractional knapsack

In [0]:
# Greedy patterns
print("GREEDY PATTERNS")
print("=" * 60)

# Pattern 1: Jump Game (can you reach the end?)
def can_jump(nums):
    """Greedy: track furthest reachable position."""
    max_reach = 0
    for i, jump in enumerate(nums):
        if i > max_reach:
            return False
        max_reach = max(max_reach, i + jump)
    return True

print("\n1. Jump Game:")
tests = [([2,3,1,1,4], True), ([3,2,1,0,4], False), ([1,0,0,0], False)]
for nums, expected in tests:
    result = can_jump(nums)
    status = "PASS" if result == expected else "FAIL"
    print(f"   [{status}] {nums} -> {result}")

# Pattern 2: Non-overlapping intervals
def erase_overlap_intervals(intervals):
    """Minimum intervals to remove to make non-overlapping."""
    if not intervals:
        return 0
    intervals.sort(key=lambda x: x[1])  # Sort by end time
    count = 0
    end = intervals[0][1]
    for i in range(1, len(intervals)):
        if intervals[i][0] < end:  # Overlap!
            count += 1  # Remove this interval
        else:
            end = intervals[i][1]  # Update end
    return count

print("\n2. Non-overlapping Intervals:")
tests = [
    ([[1,2],[2,3],[3,4],[1,3]], 1),
    ([[1,2],[1,2],[1,2]], 2),
    ([[1,2],[2,3]], 0),
]
for intervals, expected in tests:
    result = erase_overlap_intervals(intervals)
    status = "PASS" if result == expected else "FAIL"
    print(f"   [{status}] {intervals} -> remove {result}")

# Pattern 3: Gas Station
def can_complete_circuit(gas, cost):
    """Find starting station for circular route."""
    if sum(gas) < sum(cost):
        return -1
    tank = 0
    start = 0
    for i in range(len(gas)):
        tank += gas[i] - cost[i]
        if tank < 0:
            start = i + 1
            tank = 0
    return start

print("\n3. Gas Station:")
gas = [1, 2, 3, 4, 5]
cost = [3, 4, 5, 1, 2]
print(f"   gas={gas}, cost={cost}")
print(f"   Start at station: {can_complete_circuit(gas, cost)}")


---
# Summary

## Key Takeaways

1. **Sorting**: Use Python's `sorted()` in practice. Know merge sort and quick sort conceptually.
2. **Binary search**: 3 templates -- exact, lower bound, binary search on answer
3. **Binary search on answer**: "Find minimum X such that condition(X) is True"
4. **Greedy**: Works when local optimal = global optimal. Prove it or use DP instead.

## LeetCode Problems

**Easy:** Binary Search (#704), First Bad Version (#278), Best Time to Buy and Sell Stock (#121)

**Medium:** Search in Rotated Sorted Array (#33), Koko Eating Bananas (#875), Jump Game (#55)

---
*DSA Notebook 07 of 8*